In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)


In [ ]:
# Task 2: Write your code here:
df_food.head()

In [ ]:
# Task 3: Write your code here:

df_food.info()

In [ ]:
# Task 4: Write your code here:
df_food.describe()

In [ ]:
# Task 5: Write your code here:
# Price distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_food.drop(columns=['Order_ID'])

In [ ]:

df_food['Courier_Experience_yrs']= df_food['Courier_Experience_yrs'].fillna('unknown')
df_food['Delivery_Time']=df_food['Delivery_Time'].fillna('unknown')
df_food['Time_of_Day']=df_food['Time_of_Day'].fillna('unknown')
df_food['Traffic_Level']=df_food['Traffic_Level'].fillna('unknown')


In [ ]:
# Task 3: Write your code here:
df_food.drop_duplicates(inplace=True)

In [ ]:
# Task 4: Write your code here:
# Encode categorical columns - converts text to integers
for col in df_food:
 le = LabelEncoder()
 df_food[col] = le.fit_transform(df_food[col].astype(str))
df_food.head()

In [ ]:
X = df_food.drop(columns=['Delivery_Time'])
y = df_food['Delivery_Time']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# Task 6: Write your code here:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
def check_target_imbalance(df, target_column):
 print("Target Distribution:")
 print(df_food[target_column].value_counts(normalize=True))
 sns.countplot(x=df_food[target_column])
 plt.title("Target Distribution")
 plt.show()
check_target_imbalance(df_food, "Delivery_Time")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

for train_idx, val_idx in kf.split(X_scaled):
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]


In [ ]:

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    mae = mean_absolute_error(y_val, preds)
    mae_scores.append(mae)
print(f"Average MAE across folds: {np.mean(mae_scores):.2f} minutes")

In [ ]:
# Task 1: Write your code here:
feature_importance = model.feature_importances_
feature_names = X.columns

imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10,6))
plt.barh(imp_df['Feature'][:10], imp_df['Importance'][:10])
plt.gca().invert_yaxis()
plt.title("Top 10 Feature Importances")
plt.show()


In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(8,4))
plt.hist(preds, bins=40, edgecolor='black')
plt.title("Predicted Delivery Time Distribution")
plt.xlabel("Predicted Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.show()


In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor
import numpy as np

# KFold setup
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kf.split(X_scaled):
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Model 1: Random Forest
    rf = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    # Model 2: CatBoost
    cb = CatBoostRegressor(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        loss_function="MAE",
        verbose=0,
        random_state=42
    )

    # Train both models
    rf.fit(X_train, y_train)
    cb.fit(X_train, y_train)

    # Predictions
    preds_rf = rf.predict(X_val)
    preds_cb = cb.predict(X_val)

    # Average predictions
    avg_preds = (preds_rf + preds_cb) / 2

    # MAE evaluation
    mae = mean_absolute_error(y_val, avg_preds)
    mae_scores.append(mae)

# Final result
print(f"Ensemble Average MAE across folds: {np.mean(mae_scores):.2f} minutes")
